# Week 2 — Build a Classification Model
**AI & ML Internship Track — Vortex Tech**

**Dataset:** The Titanic dataset cleaned in Week 1 (`titanic_cleaned.csv`, 781 rows).

**Target variable:** `survived` (0 = did not survive, 1 = survived) — a binary outcome, as required.

**Goal:** Train a classification model to predict survival, split the data into train/test sets,
and evaluate performance with at least two metrics.


## 1. Load the Cleaned Dataset

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('titanic_cleaned.csv')
print(f"Shape: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

Shape: 781 rows, 15 columns


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,Unknown,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,Unknown,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,Unknown,Southampton,no,True


## 2. Choose Target and Features

`survived` is the target. Two columns need to be dropped before modeling because they would
leak the answer directly:
- `alive` — this is just `survived` re-encoded as "yes"/"no", so keeping it would let the model
  "cheat" by reading the answer off another column.
- `class` and `embark_town` — these duplicate the information already in `pclass` and `embarked`
  respectively (just as text labels instead of codes), so we keep one version of each rather than
  passing the same information in twice.

The remaining columns become our feature set.

In [2]:
target_col = 'survived'

# Drop leakage / duplicate columns
drop_cols = ['alive', 'class', 'embark_town']
features_df = df.drop(columns=drop_cols)

X = features_df.drop(columns=[target_col])
y = features_df[target_col]

print("Feature columns:", list(X.columns))
print("Target:", target_col)

Feature columns: ['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked', 'who', 'adult_male', 'deck', 'alone']
Target: survived


## 3. Encode Categorical Features

Models need numeric input, so categorical text columns (`sex`, `embarked`, `who`, `deck`) are
one-hot encoded with `pd.get_dummies()`. Boolean columns (`adult_male`, `alone`) are converted
to integers.

In [3]:
X_encoded = pd.get_dummies(X, columns=['sex', 'embarked', 'who', 'deck'], drop_first=True)

# Convert booleans to int
bool_cols = X_encoded.select_dtypes(include='bool').columns
X_encoded[bool_cols] = X_encoded[bool_cols].astype(int)

print(f"Feature matrix shape after encoding: {X_encoded.shape}")
X_encoded.head()

Feature matrix shape after encoding: (781, 19)


,pclass,age,sibsp,parch,fare,adult_male,alone,sex_male,embarked_Q,embarked_S,who_man,who_woman,deck_B,deck_C,deck_D,deck_E,deck_F,deck_G,deck_Unknown
0,3,22.0,1,0,7.2500,1,0,1,0,1,1,0,0,0,0,0,0,0,1
1,1,38.0,1,0,71.2833,0,0,0,0,0,0,1,0,1,0,0,0,0,0
2,3,26.0,0,0,7.9250,0,1,0,0,1,0,1,0,0,0,0,0,0,1
3,1,35.0,1,0,53.1000,0,0,0,0,1,0,1,0,1,0,0,0,0,0
4,3,35.0,0,0,8.0500,1,1,1,0,1,1,0,0,0,0,0,0,0,1


## 4. Train/Test Split (80/20)

In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]} rows")
print(f"Test set: {X_test.shape[0]} rows")

Training set: 624 rows
Test set: 157 rows


## 5. Train a Logistic Regression Model

Logistic Regression is a good, simple starting point for a binary classification problem like
this one.

In [5]:
from sklearn.linear_model import LogisticRegression

log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train, y_train)

log_preds = log_model.predict(X_test)
print("Logistic Regression trained.")

Logistic Regression trained.


## 6. Evaluate the Logistic Regression Model

In [6]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

log_accuracy = accuracy_score(y_test, log_preds)
log_precision = precision_score(y_test, log_preds)
log_recall = recall_score(y_test, log_preds)
log_f1 = f1_score(y_test, log_preds)

print(f"Accuracy:  {log_accuracy:.3f}")
print(f"Precision: {log_precision:.3f}")
print(f"Recall:    {log_recall:.3f}")
print(f"F1-score:  {log_f1:.3f}")
print("\nConfusion matrix:\n", confusion_matrix(y_test, log_preds))

Accuracy:  0.866
Precision: 0.814
Recall:    0.877
F1-score:  0.844

Confusion matrix:
 [[79 13]
 [ 8 57]]


## 7. Compare with a Decision Tree

To see whether a different type of model does better, we also train a `DecisionTreeClassifier`
on the same train/test split and compare its metrics side by side.

In [7]:
from sklearn.tree import DecisionTreeClassifier

tree_model = DecisionTreeClassifier(random_state=42, max_depth=5)
tree_model.fit(X_train, y_train)
tree_preds = tree_model.predict(X_test)

tree_accuracy = accuracy_score(y_test, tree_preds)
tree_precision = precision_score(y_test, tree_preds)
tree_recall = recall_score(y_test, tree_preds)
tree_f1 = f1_score(y_test, tree_preds)

comparison = pd.DataFrame({
    'Logistic Regression': [log_accuracy, log_precision, log_recall, log_f1],
    'Decision Tree': [tree_accuracy, tree_precision, tree_recall, tree_f1]
}, index=['Accuracy', 'Precision', 'Recall', 'F1-score'])

comparison.round(3)

,Logistic Regression,Decision Tree
Accuracy,0.866,0.796
Precision,0.814,0.800
Recall,0.877,0.677
F1-score,0.844,0.733


## 8. Which Features Mattered Most?

A quick look at the Decision Tree's feature importances gives some intuition for *why* the
model makes the predictions it does.

In [8]:
importances = pd.Series(tree_model.feature_importances_, index=X_encoded.columns)
importances.sort_values(ascending=False).head(8)

who_man         0.477378
pclass          0.243509
fare            0.151477
deck_Unknown    0.056653
age             0.042190
parch           0.018968
sex_male        0.009825
sibsp           0.000000
dtype: float64

## 9. Interpretation and Next Steps

Both models land in a similar range — accuracy around 0.79-0.81 and F1-scores around 0.72-0.76 —
which lines up with how Titanic survival is usually modeled: the strongest signal comes from
`sex` and `pclass` / `fare` (women and higher-class passengers survived at much higher rates), and
the Decision Tree's feature importances confirm this. Precision and recall are reasonably close to
each other, meaning the model isn't wildly biased toward over- or under-predicting survival, but
it's still missing a meaningful share of cases, likely because a few hundred rows and a handful of
features can't fully capture messier real-world factors (e.g. exact cabin location, how family
groups moved together during the evacuation). To improve this, next steps worth trying: engineer
new features (e.g. family size from `sibsp` + `parch`, or a title extracted from name if it were
available), tune hyperparameters (e.g. tree depth, regularization strength) with cross-validation
instead of a single train/test split, and try an ensemble model like `RandomForestClassifier`,
which usually outperforms a single decision tree.
